# 🪄 AUTO SOLVER — ปุ่มเดียวจบ (สำหรับคนที่อยากง่ายที่สุด)

โน้ตบุ๊กนี้ "เดาเอง" ว่าโจทย์เป็นงานประเภทไหน แล้วเลือกวิธี + เทรน + สร้าง submission ให้อัตโนมัติ
คุณแก้แค่ `ที่เดียว` (ชื่อ competition หรืออัปโหลดไฟล์) แล้วกดรันทีละเซลล์จากบนลงล่าง

ทำอัตโนมัติให้ได้: จำแนกรูป (image), จำแนกข้อความ (text), ตาราง-จำแนก/ทำนายตัวเลข (tabular/regression), สัญญาณ/พยากรณ์ (ทำเป็นตาราง)
ทำให้ไม่ได้ (จะบอกให้ไปเปิดโน้ตบุ๊กเฉพาะ): ตัดคำ, segmentation, บรรยายรูป/VQA, detection -- พวกนี้ฟอร์แมตพิเศษ

ขั้นตอน: (1) ติดตั้ง (2) ใส่ข้อมูล/ชื่อ comp (3) กดรันเซลล์ AUTO (4) ส่ง -- จบ


## ขั้นที่ 1 — ติดตั้ง (กดรันรอ ~5-10 นาทีครั้งแรก)

In [ ]:
!pip -q install -U "autogluon.tabular[all]" "autogluon.multimodal" pandas numpy pillow

## ขั้นที่ 2 — เอาข้อมูลเข้า (เลือก 1 ใน 3 วิธี) + เช็ค GPU

เซลล์ล่างรองรับ 3 วิธี แก้แค่ตัวแปรบนสุดให้ตรงกับวิธีที่ใช้:

วิธี A (แนะนำ) ดาวน์โหลดจาก Kaggle อัตโนมัติ: ต้องมี `kaggle.json` (kaggle.com -> รูปโปรไฟล์ -> Settings -> Account -> Create New Token)
- บน `Kaggle Notebook`: ข้อมูลต่อไว้ที่ `/kaggle/input/...` แล้ว ไม่ต้องใส่ token
- บน `Colab/เครื่อง`: ใส่ `KAGGLE_USERNAME` + `KAGGLE_KEY`

วิธี B โหลดข้อมูลมาเครื่องตัวเองแล้วอัปขึ้น Colab เอง: ลากไฟล์ (เช่น `data.zip`) ไปวางในแถบ Files ซ้ายมือของ Colab
แล้วตั้ง `DATA_DIR = "/content"` (หรือโฟลเดอร์ที่วาง) -> เซลล์จะแตก zip ให้เอง ไม่ต้องใช้ token

วิธี C ต่อ Google Drive: รัน `from google.colab import drive; drive.mount('/content/drive')` ก่อน แล้วตั้ง `DATA_DIR = "/content/drive/MyDrive/โฟลเดอร์ของคุณ"`

เซลล์นี้เช็ค GPU ให้ด้วย ถ้างานเป็น deep learning (รูป/ข้อความ/สัญญาณดิบ) ควรเปิด GPU: เมนู `Runtime > Change runtime type > T4 GPU`

In [ ]:
import os, glob, subprocess

# ----- วิธี A: ดาวน์โหลดจาก Kaggle -----
COMP = "ใส่-slug-ของ-competition-ตรงนี้"      # << แก้: slug ท้าย URL เช่น kaggle.com/competitions/titanic -> "titanic"  (ใช้เฉพาะวิธี A)
KAGGLE_USERNAME = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "0a1b2c3d4e5f..." (คีย์ยาว ~32 ตัวจาก kaggle.json)
# ----- วิธี B/C: อัปโหลดเอง / ต่อ Google Drive -----
DATA_DIR = ""          # << แก้: ถ้าอัปโหลดข้อมูลเอง ใส่ path โฟลเดอร์ เช่น "/content" (วิธี B) หรือ "/content/drive/MyDrive/data" (วิธี C) ; ใช้ Kaggle (วิธี A) ปล่อยว่าง

# เช็ค GPU (ถ้าไม่มี + งานเป็น deep learning -> เปิดที่เมนู Runtime > Change runtime type > T4 GPU)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (งาน deep learning จะช้า; งานตาราง/ตัดคำ ไม่ต้องใช้ GPU ก็ได้)")
except Exception:
    pass

def get_data(comp):
    # วิธี B/C: ผู้ใช้ตั้ง DATA_DIR เอง (อัปโหลด/ต่อ Drive) -- แตก zip ในโฟลเดอร์นั้นให้ด้วยถ้ามี
    if DATA_DIR and os.path.isdir(DATA_DIR):
        for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            subprocess.run(["unzip", "-o", "-q", z, "-d", DATA_DIR])
        print("ใช้ข้อมูลจากโฟลเดอร์ที่ตั้งเอง:", DATA_DIR); return DATA_DIR
    # บน Kaggle Notebook ข้อมูลต่อไว้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle ข้อมูลอยู่ที่", kpath); return kpath
    # วิธี A: ดาวน์โหลดด้วย Kaggle API
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("\nไฟล์ที่มี (ดูชื่อไฟล์/โฟลเดอร์ แล้วแก้ในเซลล์ถัดไปให้ตรง):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — กดรัน AUTO (เดา+เทรน+สร้าง submission ให้อัตโนมัติ)

ปกติไม่ต้องแก้อะไรในเซลล์นี้ มันจะ print ว่าเดาว่าเป็นงานอะไร + คอลัมน์ไหน ให้เช็คได้
ถ้าเดาผิด ค่อยตั้ง `FORCE_TASK` / `METRIC` ที่บรรทัดบนสุดของเซลล์

In [ ]:
import os, glob, pandas as pd, numpy as np
IMG_EXT = (".jpg",".jpeg",".png",".bmp")

# ===== ปรับเฉพาะถ้า auto เดาผิด (ปกติปล่อยไว้) =====
FORCE_TASK = None    # << ใส่ได้: "image" / "text" / "tabular" / "regression"
METRIC     = None    # << ใส่ "roc_auc" ถ้าโจทย์วัด AUC (จะได้ส่งความน่าจะเป็น)

def _find(name):
    h = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True); return h[0] if h else None
def _img_dir(keyword):
    cand = [d for d,_,fs in os.walk(DATA_DIR) if keyword in d.lower()
            and any(f.lower().endswith(IMG_EXT) for f in fs)]
    return max(cand, key=lambda d: len(os.listdir(d))) if cand else None

sample = pd.read_csv(_find("sample_submission.csv"))
ID_COL, ANS_COL = sample.columns[0], sample.columns[1]
train = pd.read_csv(_find("train.csv")) if _find("train.csv") else None
test  = pd.read_csv(_find("test.csv"))  if _find("test.csv")  else None
train_img, test_img = _img_dir("train"), _img_dir("test")
print("ไฟล์ส่งต้องมีคอลัมน์:", list(sample.columns), "| id =", ID_COL, "| คำตอบ =", ANS_COL)
for _c in list(sample.columns)[1:]:
    print(f"  - เติม '{_c}': ชนิด {sample[_c].dtype}, ตัวอย่างค่าใน sample = {list(sample[_c].dropna().unique()[:3])}")

# ดึง metric จาก Kaggle เพื่อรู้ว่าต้องส่ง ป้าย/ความน่าจะเป็น/ตัวเลข (ถ้าดึงไม่ได้ -> เดาจากข้อมูล)
_metric = None
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    _api = KaggleApi(); _api.authenticate()
    for _co in _api.competitions_list(search=COMP):
        if str(getattr(_co, "ref", "")).rstrip("/").split("/")[-1] == COMP:
            _metric = getattr(_co, "evaluationMetric", None) or getattr(_co, "evaluation_metric", None); break
except Exception:
    pass
_ml = (_metric or "").lower()
WANT_PROB = (METRIC in ("roc_auc","auc")) or any(k in _ml for k in ["auc","roc","log loss","logloss","brier","probab"])
MULTICOL  = len(sample.columns) > 2   # submission หลายคอลัมน์ = ส่งความน่าจะเป็นต่อคลาส
print("Metric (จาก Kaggle):", _metric or "ดึงไม่ได้ (เปิด tab Evaluation อ่านเอง)",
      "| auto จะส่งเป็น:", "ความน่าจะเป็น" if (WANT_PROB or MULTICOL) else "ป้าย/ตัวเลข")

assert train is not None, "หา train.csv ไม่เจอ -> เช็ค path/ใช้โน้ตบุ๊กเฉพาะ"

# ----- กันงานฟอร์แมตพิเศษ (submission ไม่ใช่ 1 แถวต่อ 1 ตัวอย่าง test) -----
if test is not None and len(sample) != len(test) and not (train_img and test_img):
    raise SystemExit("งานนี้ submission ไม่ใช่ 1 แถวต่อ 1 แถวของ test (น่าจะเป็นตัดคำ/segmentation) "
                     "-> เปิด 00_MASTER_ROUTER.ipynb พิมพ์โจทย์ แล้วใช้โน้ตบุ๊กเฉพาะ")

# ----- เดาคอลัมน์เป้าหมาย (คอลัมน์ที่อยู่ใน train แต่ไม่อยู่ใน test) -----
def detect_target():
    if test is not None:
        cand = [c for c in train.columns if c not in test.columns and c != ID_COL]
        if len(cand) == 1: return cand[0]
    if ANS_COL in train.columns: return ANS_COL
    return [c for c in train.columns if c != ID_COL][-1]

# ----- เดาประเภทงาน -----
def detect_task():
    if FORCE_TASK: return FORCE_TASK
    if train_img and test_img:
        imgcols = [c for c in train.columns if train[c].astype(str).str.lower().str.endswith(IMG_EXT).mean() > 0.5]
        if imgcols: return "image"
    textcols = [c for c in train.columns if train[c].dtype == object and c != ID_COL
                and train[c].astype(str).str.len().mean() >= 10]
    if textcols: return "text"
    tgt = detect_target()
    if pd.api.types.is_numeric_dtype(train[tgt]) and train[tgt].nunique() > 20: return "regression"
    return "tabular"

task = detect_task()
print("ตรวจพบว่าเป็นงาน:", {"image":"จำแนกรูป","text":"จำแนกข้อความ","tabular":"ตาราง-จำแนกคลาส",
                          "regression":"ตาราง-ทำนายตัวเลข"}.get(task, task))

out = sample.copy()
if task == "image":
    from autogluon.multimodal import MultiModalPredictor
    IMG_COL = [c for c in train.columns if train[c].astype(str).str.lower().str.endswith(IMG_EXT).mean() > 0.5][0]
    LABEL_COL = [c for c in train.columns if c not in (IMG_COL, ID_COL)][0]
    exts = [os.path.splitext(f)[1] for f in os.listdir(test_img) if f.lower().endswith(IMG_EXT)]
    ext = exts[0] if exts else ".jpg"
    print("  รูป:", IMG_COL, "| ป้าย:", LABEL_COL, "| นามสกุลไฟล์ test:", ext)
    tr = train.copy(); tr["image"] = tr[IMG_COL].apply(lambda n: os.path.join(train_img, str(n)))
    p = MultiModalPredictor(label=LABEL_COL, path="ag_auto").fit(tr[["image", LABEL_COL]], time_limit=900)
    td = sample.copy(); td["image"] = td[ID_COL].apply(lambda i: os.path.join(test_img, str(i)+ext))
    out[ANS_COL] = p.predict(td[["image"]]).values

elif task == "text":
    from autogluon.multimodal import MultiModalPredictor
    TEXT_COL = [c for c in train.columns if train[c].dtype == object and c != ID_COL
                and train[c].astype(str).str.len().mean() >= 10][0]
    LABEL_COL = detect_target()
    print("  ข้อความ:", TEXT_COL, "| ป้าย:", LABEL_COL)
    p = MultiModalPredictor(label=LABEL_COL, path="ag_auto").fit(train[[TEXT_COL, LABEL_COL]], time_limit=600)
    out[ANS_COL] = p.predict(test[[TEXT_COL]]).values

else:  # tabular / regression
    from autogluon.tabular import TabularPredictor
    LABEL_COL = detect_target()
    is_reg = (task == "regression")
    print("  เป้าหมาย:", LABEL_COL, "| ชนิด:", "ทำนายตัวเลข" if is_reg else "จำแนกคลาส")
    tr = train.drop(columns=[c for c in [ID_COL] if c in train.columns])
    te = test.drop(columns=[c for c in [ID_COL] if c in test.columns])
    p = TabularPredictor(label=LABEL_COL, problem_type=("regression" if is_reg else None),
                         path="ag_auto").fit(tr, presets="best_quality", time_limit=600)
    if is_reg:
        out[ANS_COL] = p.predict(te).values
    elif MULTICOL:
        # submission หลายคอลัมน์ = ความน่าจะเป็นต่อคลาส -> เติมแต่ละคอลัมน์ให้ตรงชื่อคลาส
        proba = p.predict_proba(te); name_map = {str(cc): cc for cc in proba.columns}
        for col in list(sample.columns[1:]):
            key = name_map.get(str(col)) or name_map.get(str(col).split("_")[-1])
            out[col] = proba[key].values if key is not None else 0.0
        print("  (เติมความน่าจะเป็นต่อคลาส", len(sample.columns)-1, "คอลัมน์)")
    elif WANT_PROB and getattr(p, "positive_class", None) is not None:
        out[ANS_COL] = p.predict_proba(te)[p.positive_class].values   # ส่งความน่าจะเป็น (AUC/log-loss)
    else:
        out[ANS_COL] = p.predict(te).values
        # งานจำแนก 2 คลาส: เซฟไฟล์ความน่าจะเป็นไว้ด้วย เผื่อ metric เป็น AUC
        if getattr(p, "positive_class", None) is not None and train[LABEL_COL].nunique() == 2:
            o2 = sample.copy(); o2[ANS_COL] = p.predict_proba(te)[p.positive_class].values
            o2.to_csv("submission_proba.csv", index=False)
            print("  (เซฟ submission_proba.csv ไว้ด้วย เผื่อ metric เป็น AUC)")

out.to_csv("submission.csv", index=False)
print("\nเสร็จ! บันทึก submission.csv", out.shape);
try: display(out.head())
except Exception: print(out.head())

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
import pandas as pd, glob, os
FILE = "submission.csv"   # << แก้เป็นไฟล์ที่จะส่ง เช่น "submission_lgbm.csv" หรือ "submission_timm.csv"
# ตรวจรูปแบบก่อนส่งอัตโนมัติ (กันแก้ config ผิดแล้วส่งฟอร์แมตเพี้ยน -> ได้ 0 คะแนน)
_sub = pd.read_csv(FILE)
_sam = glob.glob(os.path.join(DATA_DIR, "*ample*ubmission*.csv"))
if _sam:
    _s = pd.read_csv(_sam[0])
    print("คอลัมน์ตรง sample:", list(_sub.columns)==list(_s.columns), "| จำนวนแถวตรง:", len(_sub)==len(_s))
    assert list(_sub.columns)==list(_s.columns), f"คอลัมน์ไม่ตรง sample! {list(_sub.columns)} != {list(_s.columns)} -> แก้ก่อนส่ง"
print("พร้อมส่ง:", FILE, _sub.shape)
!kaggle competitions submit -c {COMP} -f {FILE} -m "auto solver"
!kaggle competitions submissions -c {COMP} | head